# 08 -- Building modal

Transfer functions, coherence, mode shapes and damping from the vertical array.

In [ ]:
import sys
sys.path.insert(0, "../src")          # run from examples/

from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"   # folder with the .h5 files

In [ ]:
ds = SensorDataset(path=DATA, date_source="filename", load_mode="auto", verbose=True)
ds.verbose  = True      # internal prints, ShakerMakerResults style
ds.n_jobs   = 4
ds.parallel = True
ds.devices

In [ ]:
# Set the sensor geometry up front. Until you have surveyed UTM values you can
# put approximate ones here; this overrides config.SENSOR_GEOMETRY for the
# session. Only relative positions matter (plan distances, height differences).
settings.SENSOR_GEOMETRY["MOF00134"].update(E=500000.0, N=6300000.0, elev= 0.0, azimuth=0.0)   # base -1
settings.SENSOR_GEOMETRY["MNAT0031"].update(E=500000.0, N=6300000.0, elev= 7.0, azimuth=0.0)   # floor 2
settings.SENSOR_GEOMETRY["MNAT0034"].update(E=500000.0, N=6300000.0, elev=10.5, azimuth=0.0)   # floor 3
settings.SENSOR_GEOMETRY["MOF00135"].update(E=500000.0, N=6300000.0, elev=14.0, azimuth=0.0)   # floor 4 (stairwell)
settings.SENSOR_GEOMETRY["MOF00136"].update(E=500006.0, N=6300004.0, elev=14.0, azimuth=0.0)   # floor 4 (cantilever)
settings.SENSOR_GEOMETRY

## Transfer function: one pair and the full stack

In [ ]:
frf  = ds.transfer_function(numerator="MOF00135", denominator="MOF00134", component="x",
                            estimator="H1", nperseg=1024, noverlap=512, smooth="konno", bexp=40, fmax=25.0)
frfs = ds.transfer_function(base="MOF00134", floors="all", component="x")   # base -> p2, p3, p4
frf["modal_freqs"]

## Coherence (validate the peaks)

In [ ]:
ds.coherence("MOF00135", "MOF00134", component="x", nperseg=1024)
ds.coherence_matrix(component="x")

## Modal frequencies, mode shapes, damping

In [ ]:
ds.modal_frequencies(component="x")
shapes = ds.mode_shapes(component="x")     # amplitude/phase per floor, ordered by height
ds.MOF00135.modal_tracking(component="x", window="10min", overlap=0.5, fband=(1.0,8.0), n_modes=2)